# MedGeemaSOAP: Fine-Tuning MedGemma 1.5 4B-IT on Kaggle

This notebook runs the QLoRA fine-tuning for the MedGeemaSOAP project. It is optimized for Kaggle (T4 GPU) with RAM safeguards.

## Workflow
1.  **Setup Environment**: Install dependencies and authenticate with HuggingFace.
2.  **Download & Cache (Run Once)**: Download the base model to Kaggle working directory.
3.  **Train (Re-runnable)**: Load the model and start/resume training.

## Setup Instructions
1.  **Accelerator**: Enable GPU (Settings → Accelerator → GPU T4 x2).
2.  **Dataset**: Upload your training data as a Kaggle Dataset and attach it to this notebook.
3.  **HuggingFace Token**: Add HUGGING_FACE_HUB_TOKEN to Kaggle Secrets (Settings → Secrets).

In [ ]:
# VERSION CHECK
print("Notebook Version: 4.0 (MedGemma 1.5 4B-IT - Kaggle)")
print("If you do not see this message, reload the notebook file!")

In [ ]:
# 1. Environment Setup (MUST RUN FIRST - before any torch import!)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 🔥 CRITICAL: Single GPU mode
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/kaggle/working/.cache/huggingface"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
print("✅ Environment set (Single GPU mode)")

In [ ]:
# 2. Install Dependencies using uv (MUST RESTART KERNEL after running this cell!)
!pip install -q uv

# Install all packages using uv
!uv pip install --system --extra-index-url https://download.pytorch.org/whl/cu118 \
    torch torchvision \
    "trl==0.27.1" "peft==0.18.1" bitsandbytes \
    "numpy<2.0" scipy scikit-learn transformers \
    "Pillow>=9.5.0,<10" "setuptools<81" datasets accelerate huggingface-hub

print("\n" + "="*70)
print("⚠️  IMPORTANT: RESTART THE KERNEL NOW!")
print("="*70)
print("After the kernel restarts, skip this cell and continue with Cell 5.")
print("="*70)

In [1]:
# 3. Check GPU and Libraries (torch import happens AFTER env vars are set)
import torch
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA devices visible: {torch.cuda.device_count()}")  # Should be 1
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Torch version: 2.8.0+cu126
CUDA available: True
CUDA devices visible: 2
GPU: Tesla T4


In [2]:
# 5. Login to Hugging Face (using Kaggle Secrets)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")
os.environ["HF_TOKEN"] = secret_value_0
login(token=secret_value_0)
print("✅ HuggingFace authenticated")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HuggingFace authenticated


In [3]:
# 6. Configuration
import torch
from peft import LoraConfig, TaskType
from transformers import BitsAndBytesConfig

# QLoRA Config for Cell 2
QLORA_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float32,  # Use fp32 to avoid bf16 conflicts on T4
)

LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
print("✅ Configuration ready")

2026-02-03 13:55:07.394430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770126907.592116     432 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770126907.647329     432 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770126908.112965     432 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770126908.113023     432 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770126908.113026     432 computation_placer.cc:177] computation placer alr

✅ Configuration ready


In [ ]:
# # UTILITY: Inspect and safely clear /kaggle/working
# """
# Usage:
# - Run this cell to list free space and the top N largest files/directories under /kaggle/working.
# - To delete specific paths, set DELETE_PATHS to a list of absolute paths and set
#   CONFIRM_DELETE = "DELETE SELECTED" then re-run this cell.
# - To wipe everything under /kaggle/working (dangerous), set CONFIRM_DELETE = "YES DELETE ALL" and re-run.

# This cell requires an explicit confirmation string to perform deletions to avoid accidents.
# """

# import shutil, os, heapq, pathlib

# ROOT = pathlib.Path("/kaggle/working")

# # --- Configuration (edit these and re-run to act) ---
# DELETE_PATHS = []  # e.g. ["/kaggle/working/models/old-model", "/kaggle/working/medgeema-soap-lora.zip"]
# CONFIRM_DELETE = "YES DELETE ALL"  # set to "DELETE SELECTED" or "YES DELETE ALL" to perform deletions
# TOP_N = 30

# # --- Helpers ---
# def human_size(num_bytes):
#     for unit in ['B','KB','MB','GB','TB']:
#         if abs(num_bytes) < 1024.0:
#             return f"{num_bytes:3.1f}{unit}"
#         num_bytes /= 1024.0
#     return f"{num_bytes:.1f}PB"


# def disk_usage(path=ROOT):
#     du = shutil.disk_usage(str(path))
#     return du


# def top_files(path=ROOT, n=TOP_N):
#     heap = []
#     path = pathlib.Path(path)
#     for root, dirs, files in os.walk(path):
#         for name in files:
#             try:
#                 fp = pathlib.Path(root) / name
#                 s = fp.stat().st_size
#             except Exception:
#                 continue
#             if len(heap) < n:
#                 heapq.heappush(heap, (s, str(fp)))
#             else:
#                 heapq.heappushpop(heap, (s, str(fp)))
#     return sorted(heap, reverse=True)


# # --- Report ---
# du = disk_usage(ROOT)
# print(f"Disk usage for {ROOT} — total={human_size(du.total)}, used={human_size(du.used)}, free={human_size(du.free)}")

# print(f"\nTop {TOP_N} largest files under {ROOT}:")
# for s, p in top_files(ROOT, TOP_N):
#     print(f"{human_size(s):>8}  {p}")

# # --- Deletion actions (explicit) ---
# if CONFIRM_DELETE == "YES DELETE ALL":
#     print("\nCONFIRMED: Wiping all children under /kaggle/working (this may take a while)")
#     for child in ROOT.iterdir():
#         try:
#             if child.is_dir():
#                 shutil.rmtree(child)
#             else:
#                 child.unlink()
#             print("Deleted", child)
#         except Exception as e:
#             print("Failed to delete", child, type(e).__name__, str(e))
#     du = disk_usage(ROOT)
#     print(f"\nDone. New free space: {human_size(du.free)}")

# elif CONFIRM_DELETE == "DELETE SELECTED" and DELETE_PATHS:
#     print("\nCONFIRMED: Deleting selected paths")
#     for p in DELETE_PATHS:
#         try:
#             tp = pathlib.Path(p)
#             if not tp.exists():
#                 print("Not found:", tp)
#                 continue
#             if tp.is_dir():
#                 shutil.rmtree(tp)
#             else:
#                 tp.unlink()
#             print("Deleted", tp)
#         except Exception as e:
#             print("Failed to delete", p, type(e).__name__, str(e))
#     du = disk_usage(ROOT)
#     print(f"\nDone. New free space: {human_size(du.free)}")

# elif CONFIRM_DELETE in ("DELETE SELECTED", "YES DELETE ALL"):
#     # Cases where confirmation string matches but no paths provided
#     if CONFIRM_DELETE == "DELETE SELECTED" and not DELETE_PATHS:
#         print('\nNo DELETE_PATHS provided. Set DELETE_PATHS to a list of absolute paths to delete and re-run.')
# else:
#     print('\nNo destructive action taken. To delete, set CONFIRM_DELETE and re-run this cell.')

In [4]:
# CELL 1: Download base model to Kaggle working directory
from huggingface_hub import snapshot_download
from pathlib import Path
import shutil

BASE_MODEL = "google/medgemma-1.5-4b-it"
CACHE_DIR = "/kaggle/working/models/medgemma-1.5-4b-it"

# Create directory
path_obj = Path(CACHE_DIR)
path_obj.mkdir(parents=True, exist_ok=True)

# Quick disk-space check
REQUIRED_GB = 15
free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)
print(f"Free space: {free_gb:.1f} GB")
if free_gb < REQUIRED_GB:
    raise RuntimeError(
        f"Insufficient free disk space ({free_gb:.1f} GB). Need at least {REQUIRED_GB} GB. "
        "Please free space or change CACHE_DIR to a larger storage location before proceeding."
    )

print(f"⬇️ Downloading {BASE_MODEL} to {CACHE_DIR}...")

# snapshot_download is smart: verifies integrity, resumes if interrupted
snapshot_download(
    repo_id=BASE_MODEL,
    local_dir=CACHE_DIR,
    local_dir_use_symlinks=False,
    resume_download=True,
    ignore_patterns=["*.msgpack", "*.h5", "*.ot"]  # Skip unnecessary files
)

# Verify files are present
model_files = list(Path(CACHE_DIR).glob("**/*.safetensors")) + list(Path(CACHE_DIR).glob("**/config.json"))
if not model_files:
    raise RuntimeError(f"Download failed - no model files found in {CACHE_DIR}")

total_size = sum(f.stat().st_size for f in model_files if f.is_file())
print(f"✅ Model downloaded successfully!")
print(f"   Files: {len(model_files)}, Size: {total_size/1024**3:.2f} GB")
print(f"   Location: {CACHE_DIR}")

Free space: 19.5 GB
⬇️ Downloading google/medgemma-1.5-4b-it to /kaggle/working/models/medgemma-1.5-4b-it...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

✅ Model downloaded successfully!
   Files: 3, Size: 8.01 GB
   Location: /kaggle/working/models/medgemma-1.5-4b-it


In [5]:
# CELL 2A: Dataset Loading & Tokenization (RAM Optimized)

import gc
import os
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer

# RAM FIXES
gc.collect()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# PATHS
CACHE_DIR = "/kaggle/working/models/medgemma-1.5-4b-it"
DATASET_DIR = "/kaggle/input/clinical-data-v2"
OUTPUT_DIR = "/kaggle/working/models/medgeema-soap-lora"

# LOAD DATA (STREAMING MODE - RAM friendly)
print("📂 Loading dataset in streaming mode...")
dataset = load_dataset("json", data_files={
    "train": f"{DATASET_DIR}/clinical_notes_train.jsonl",
    "validation": f"{DATASET_DIR}/clinical_notes_val.jsonl",
}, streaming=True)

# SANITY CHECK: Verify required fields
sample = next(iter(dataset["train"]))
REQUIRED_FIELDS = ["instruction", "input", "output"]
missing = [f for f in REQUIRED_FIELDS if f not in sample]
if missing:
    raise ValueError(f"❌ Missing required fields: {missing}. Found: {list(sample.keys())}")

print("✅ Dataset loaded. Sample preview:")
for k in REQUIRED_FIELDS:
    print(f"   • {k}: {str(sample.get(k))[:150].replace(chr(10), ' ')}...")

# TOKENIZER SETUP
print(f"\n🔤 Loading tokenizer from {CACHE_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(CACHE_DIR, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = 512  # Optimized for medical notes (increased from 384)

print(f"   • Vocab size: {tokenizer.vocab_size}")
print(f"   • Max length: {tokenizer.model_max_length}")

# TOKENIZATION FUNCTION (Gemma chat template)
def tokenize_fn(batch):
    """Convert examples to Gemma chat format and tokenize"""
    texts = []
    for ins, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        # Combine instruction and input
        user_content = f"{ins}\n\n{inp}" if inp and str(inp).strip() else str(ins)
        
        # Format as Gemma conversation
        messages = [
            {"role": "user", "content": user_content},
            {"role": "model", "content": out}
        ]
        
        # Apply Gemma chat template
        full_text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(full_text)
    
    # Tokenize with padding and truncation
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=tokenizer.model_max_length,
        padding="max_length",
        return_tensors=None,  # Return lists for streaming
    )
    
    # Copy input_ids to labels for causal LM
    tokenized["labels"] = [list(ids) for ids in tokenized["input_ids"]]
    return tokenized

# TOKENIZE DATASETS
print("\n🔄 Tokenizing datasets (this happens on-the-fly during training)...")
train_dataset = dataset["train"].map(
    tokenize_fn, 
    batched=True, 
    remove_columns=list(sample.keys())
)

val_dataset = None
try:
    val_sample = next(iter(dataset["validation"]))
    val_dataset = dataset["validation"].map(
        tokenize_fn, 
        batched=True, 
        remove_columns=list(val_sample.keys())
    )
    print("✅ Train and validation datasets tokenized")
except Exception as e:
    print(f"⚠️  Validation dataset skipped: {e}")
    print("✅ Train dataset tokenized")

print(f"\n💾 Ready for training. Output will be saved to: {OUTPUT_DIR}")

📂 Loading dataset in streaming mode...
✅ Dataset loaded. Sample preview:
   • instruction: Convert the following messy clinical note into a structured SOAP format....
   • input: 19yo M presents s/p TBI MVA 2mo. cognitive deficits memory attention. behavior changes. outpatient rehab. Δ TBI cognitive rehabilitation...
   • output: ### Subjective Patient is a 19-year-old male, 2 months status post moderate traumatic brain injury from motor vehicle accident. Was unconscious for 20...

🔤 Loading tokenizer from /kaggle/working/models/medgemma-1.5-4b-it...
   • Vocab size: 262144
   • Max length: 512

🔄 Tokenizing datasets (this happens on-the-fly during training)...
✅ Train and validation datasets tokenized

💾 Ready for training. Output will be saved to: /kaggle/working/models/medgeema-soap-lora


In [7]:
# CELL 2B: Model Training (QLoRA with Memory Optimization)

import torch
import gc
from transformers import AutoModelForCausalLM
from peft import get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# AGGRESSIVE MEMORY CLEANUP
gc.collect()
torch.cuda.empty_cache()

# LOAD MODEL WITH QLORA
print("🤖 Loading model with 4-bit quantization...")
print(f"   • Model: {CACHE_DIR}")
print(f"   • Quantization: 4-bit NF4")
print(f"   • Compute dtype: float32 (T4 optimized)")

model = AutoModelForCausalLM.from_pretrained(
    CACHE_DIR,
    quantization_config=QLORA_CONFIG,
    device_map="auto",
    low_cpu_mem_usage=True,  # Layer-by-layer loading to avoid RAM spikes
    trust_remote_code=True,
)

# MEMORY OPTIMIZATIONS
print("\n⚙️  Applying memory optimizations...")
model.gradient_checkpointing_enable()  # Trade compute for memory
model.config.use_cache = False  # Disable KV cache during training

# PREPARE FOR LORA TRAINING
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LORA_CONFIG)

print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

# TRAINING CONFIGURATION (Optimized for Kaggle T4)
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    
    # Training schedule
    num_train_epochs=3,
    max_steps=500,  # Total training steps
    
    # Batch settings (effective batch = 2 * 8 = 16)
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    
    # Optimization
    learning_rate=2e-4,
    warmup_steps=100,
    optim="adamw_torch",  # Memory-efficient optimizer
    
    # Logging & saving
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,  # Checkpoint every 200 steps
    save_total_limit=2,
    
    # Evaluation
    eval_strategy="no",  # Disable to save time/memory
    
    # Performance
    fp16=False,  # Use fp32 for T4 stability
    bf16=False,
    dataloader_num_workers=2,  # Parallel data loading
    
    # TRL-specific
    packing=False,  # Simpler, more stable
    remove_unused_columns=False,
    
    # Reporting
    report_to="none",  # Disable wandb/tensorboard
)

# CREATE TRAINER (no max_seq_length - handled by tokenizer.model_max_length)
print("\n🏋️  Initializing trainer...")
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,  # TRL 0.27.1 uses processing_class instead of tokenizer
)

# START TRAINING
print("\n" + "="*60)
print("🚀 STARTING TRAINING")
print("="*60)
print(f"📈 Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"🎯 Total steps: {training_args.max_steps}")
print(f"💾 Checkpoints every {training_args.save_steps} steps")
print(f"📏 Max sequence length: {tokenizer.model_max_length} (controlled by tokenizer)")
print("="*60 + "\n")

try:
    # Try to resume from checkpoint if exists
    trainer.train(resume_from_checkpoint=True)
except Exception as e:
    if "No valid checkpoint found" in str(e) or "resume_from_checkpoint" in str(e):
        print("⚠️  No checkpoint found, starting fresh training...")
        trainer.train()
    else:
        print(f"❌ Training error: {e}")
        raise

# SAVE FINAL MODEL
print("\n💾 Saving final model...")
trainer.save_model(f"{OUTPUT_DIR}/final")

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print(f"📁 Model saved to: {OUTPUT_DIR}/final")
print("🎉 You can now run the next cell to zip or upload the model")
print("="*60)

🤖 Loading model with 4-bit quantization...
   • Model: /kaggle/working/models/medgemma-1.5-4b-it
   • Quantization: 4-bit NF4
   • Compute dtype: float32 (T4 optimized)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


⚙️  Applying memory optimizations...

📊 Trainable parameters:
trainable params: 29,843,456 || all params: 4,329,922,928 || trainable%: 0.6892

🏋️  Initializing trainer...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.



🚀 STARTING TRAINING
📈 Effective batch size: 16
🎯 Total steps: 500
💾 Checkpoints every 200 steps
📏 Max sequence length: 512 (controlled by tokenizer)

⚠️  No checkpoint found, starting fresh training...


Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.


Step,Training Loss
25,2.001600
50,1.352100
75,1.064000
100,0.871800
125,0.653800
150,0.470400
175,0.365600
200,0.266000
225,0.201100
250,0.155200


Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping 1 dataloader workers.
Too many dataloader workers: 2 (max is dataset.num_shards=1). Stopping


💾 Saving final model...

✅ TRAINING COMPLETE!
📁 Model saved to: /kaggle/working/models/medgeema-soap-lora/final
🎉 You can now run the next cell to zip or upload the model


In [8]:
# CELL 3: Download Model as ZIP
import shutil
import os

OUTPUT_DIR = "/kaggle/working/models/medgeema-soap-lora/final"
ZIP_PATH = "/kaggle/working/medgeema-soap-lora.zip"

print("📦 Zipping model...")
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f"✅ Model zipped: {ZIP_PATH}")
print(f"📁 Size: {os.path.getsize(ZIP_PATH) / (1024*1024):.1f} MB")
print("\n👉 Download: Output panel (right side) → Click 'medgeema-soap-lora.zip' → Download")

📦 Zipping model...
✅ Model zipped: /kaggle/working/medgeema-soap-lora.zip
📁 Size: 50.9 MB

👉 Download: Output panel (right side) → Click 'medgeema-soap-lora.zip' → Download


In [ ]:
from huggingface_hub import login
import os

# Replace with your actual write-access token
token = "hf_..."  # Your new token here
os.environ["HF_TOKEN"] = token
login(token=token)
print("✅ HuggingFace authenticated with write access")

In [3]:
# CELL 4: Push to Hugging Face Hub
from huggingface_hub import HfApi, whoami
from pathlib import Path
import json

api = HfApi()

# Auto-detect your username
user_info = whoami()
username = user_info["name"]
print(f"📝 Logged in as: {username}")

REPO_NAME = "Med_scribe_V2"
repo_id = f"{username}/{REPO_NAME}"
OUTPUT_DIR = "/kaggle/working/models/medgeema-soap-lora/final"

# Fix the adapter_config.json to use correct base_model ID
config_path = Path(OUTPUT_DIR) / "adapter_config.json"
if config_path.exists():
    with open(config_path, "r") as f:
        config = json.load(f)
    # Replace local path with HuggingFace model ID
    config["base_model_name_or_path"] = "google/medgemma-1.5-4b-it"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print("✅ Fixed adapter_config.json base_model path")

# Create proper README.md
readme_content = """---
library_name: peft
base_model: google/medgemma-1.5-4b-it
tags:
- medical
- clinical-notes
- lora
- qlora
license: apache-2.0
---

# Med Scribe - Clinical Notes LoRA Adapter

Fine-tuned LoRA adapter for MedGemma 1.5 4B-IT, specialized for generating clinical notes from doctor-patient conversations.

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained("google/medgemma-1.5-4b-it")
model = PeftModel.from_pretrained(base_model, "YOUR_USERNAME/Med_scribe")
tokenizer = AutoTokenizer.from_pretrained("google/medgemma-1.5-4b-it")
```

## Training Details
- Base Model: google/medgemma-1.5-4b-it
- Method: QLoRA (4-bit quantization)
- LoRA rank: 16
- Training steps: 500
"""

readme_path = Path(OUTPUT_DIR) / "README.md"
with open(readme_path, "w") as f:
    f.write(readme_content.replace("YOUR_USERNAME", username))
print("✅ Created README.md")

print(f"📤 Uploading to: {repo_id}")
api.create_repo(repo_id, private=True, exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=repo_id,
    commit_message="Upload fine-tuned LoRA adapter"
)
print(f"✅ Model uploaded to: https://huggingface.co/{repo_id}")

📝 Logged in as: sahabajalam
✅ Fixed adapter_config.json base_model path
✅ Created README.md
📤 Uploading to: sahabajalam/Med_scribe_V2


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Model uploaded to: https://huggingface.co/sahabajalam/Med_scribe_V2
